# Training DPO for Multi-Objective Alignment


This file is a notebook demo for training our single- vs. multi-objective DPO models. <br>
Our main training tasks are run using the file `src/multi_dpo_training.py`. <br>
The complete training logs can be found under `logs/multi-dpo-training-logs/` folder. <br>
The trained model weights can be found at: https://drive.google.com/drive/u/1/folders/1aRMeu6YbQQjsO2KWWFND0a3PiQd-0s00


In [1]:
import json

filename = "ultrafeedback_swe_aligned.jsonl"

with open(filename, "r", encoding="utf-8") as f:
    for i in range(5):
        line = f.readline().strip()
        data = json.loads(line)
        print(json.dumps(data, indent=2, ensure_ascii=False))
        print("\n" + "="*80 + "\n")

{
  "prompt": "Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here's some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << \"Enter the name of a country: \";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}",
  "chosen": "Here's a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea:\n\n#include <iostream>\n#include <string>\n#include <set>\n#include <map>\n#include <algorithm>\n\nusing namespace std;\n\nint main() {\n    // store countries and their bordering seas in a map\n    map<string, set<string>> countries;\n    countries[\"Algeria\"] = {\"Mediterranean Sea\", \"North African Coast\"};\n    countries[\"France\"] = {\"Mediterranean Sea\", \"English Channel\"};

In [2]:
!pip install -U transformers

In [4]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig


# ================================
# 1. Paths and model config
# ================================

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
DATA_PATH = "ultrafeedback_swe_aligned.jsonl"
OUTPUT_DIR = "./qwen3b-dpo-swe-lora-final"


# ================================
# 2. Load tokenizer
# ================================
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side="left"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ================================
# 3. Load base model (FP16 or BF16)
# ================================
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model.gradient_checkpointing_enable()


# ================================
# 4. LoRA Configuration
# ================================
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    # r=4,
    # lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj","down_proj"
        # "gate_proj", "up_proj", "down_proj"
    ]
)


# ================================
# 5. Load DPO dataset
# ================================
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

def convert_to_conversational(example):
    return {
        "prompt": [
            {"role": "user", "content": example["prompt"]}
        ],
        "chosen": [
            {"role": "assistant", "content": example["chosen"]}
        ],
        "rejected": [
            {"role": "assistant", "content": example["rejected"]}
        ],
    }

dataset = dataset.map(convert_to_conversational)


# ================================
# 6. DPO Training Configuration
# ================================
dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    beta=0.1,
    num_train_epochs=1,
    logging_steps=20,
    # save_steps=500,
    # save_total_limit=2,
    max_steps=1000,

    bf16=True,
    max_length=1024,
    max_prompt_length=512,
    remove_unused_columns=False,
    gradient_checkpointing=True,

    report_to=[]
)


# ================================
# 7. Initialize DPO Trainer
# ================================
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dataset,
    peft_config=lora_config
)

print("Starting DPO LoRA training...\n")
print(dataset[0])
trainer.train()


# ================================
# 8. Save final model
# ================================
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nTraining completed!")
print("LoRA adapter saved to:", OUTPUT_DIR)

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.90it/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting DPO LoRA training...

{'prompt': [{'content': 'Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here\'s some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << "Enter the name of a country: ";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}', 'role': 'user'}], 'chosen': [{'content': 'Here\'s a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea:\n\n#include <iostream>\n#include <string>\n#include <set>\n#include <map>\n#include <algorithm>\n\nusing namespace std;\n\nint main() {\n    // store countries and their bordering seas in a map\n    map<string, set<string>> countries;\n    countries["Algeria"] = {"Mediterranean Sea", "North African Coast"};\n    count

/home/hice1/xyang645/.conda/envs/ibm_env/lib/python3.10/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss
20,0.692600
40,0.662600
60,0.638200
80,0.595100
100,0.563300
120,0.558200
140,0.516400
160,0.475100
180,0.475200
200,0.449300


/home/hice1/xyang645/.conda/envs/ibm_env/lib/python3.10/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(



Training completed!
LoRA adapter saved to: ./qwen3b-dpo-swe-lora-final
